# Hands-On Learning to Rank (LTR)


### Include required packages

In [61]:
import os
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.backends.cudnn as cudnn
import wandb
import datetime

In [62]:
seed = 42
torch.cuda.manual_seed(seed)
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
cudnn.deterministic = True
cudnn.benchmark = False

In [63]:
DEVICE = torch.device("cuda:0")
# DATA_DIR = '../data/MSLR-Web10K/Fold1/'
DATA_DIR = '../data/MQ2007/Fold1/'

MODE = 'pointwise'

FEAT_COUNT = 136 if 'MSLR' in DATA_DIR else 46
FEAT_SCALE = 1000
MB_SIZE = 1024
NUM_HIDDEN_NODES = 192 if 'MSLR' in DATA_DIR else 64
EPOCH_SIZE = 2048
NUM_EPOCHS = 50
LEARNING_RATE = 2e-4
DROPOUT_RATE = 0.1


DATA_FILE_TRAIN = os.path.join(DATA_DIR, 'train.txt')
DATA_FILE_TEST = os.path.join(DATA_DIR, 'train.txt')
MODEL_SAVE_PATH = f"ltr.{DATA_DIR.split('/')[-3]}.{NUM_EPOCHS}.pth"

### Configure the Weights and Biases

In [64]:
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
wandb.init(project=f"ltr_{DATA_DIR.split('/')[-3]}_wtraf", name=f"{MODE}_{timestamp}")
wandb.config.update({
    'seed': seed,
    'feat_count': FEAT_COUNT,
    'feat_scale': FEAT_SCALE,
    'mb_size': MB_SIZE,
    'num_hidden_nodes': NUM_HIDDEN_NODES,
    'epoch_size': EPOCH_SIZE,
    'num_epochs': NUM_EPOCHS,
    'learning_rate': LEARNING_RATE,
    'dropout_rate': DROPOUT_RATE,
    'data_file_train': DATA_FILE_TRAIN,
    'data_file_test': DATA_FILE_TEST,
    'model_save_path': MODEL_SAVE_PATH,
})
print(timestamp)

20241004-030417


### Define train and test data readers 

In [65]:
def parse_line(line):
    tokens = line.strip().split(' ')
    qid = -1
    feat = []
    label = int(tokens[0])
    
    for i in range(FEAT_COUNT):
        feat.append(0)
    
    for i in range(1, len(tokens)):
        sub_tokens = tokens[i].split(':')
        if sub_tokens[0] == 'qid':
            qid = int(sub_tokens[1])
        else:
            try:
                feat_idx = int(sub_tokens[0])
                feat_val = float(sub_tokens[1])
                feat[feat_idx - 1] = int(feat_val * FEAT_SCALE)
            except:
                pass
    return qid, label, feat

In [66]:
class DataReaderTrain():
    def __init__(self, data_file):
        self.data_file = data_file
        self.__load_data(self.data_file)

    def __iter__(self):
        self.__allocate_minibatch()
        return self

    def __load_data(self, data_file):
        self.data = {}
        with open(data_file, mode='r', encoding="utf-8") as f:
            for line in f:
                qid, label, feat = parse_line(line)
                if qid not in self.data:
                    self.data[qid] = {}
                if label not in self.data[qid]:
                    self.data[qid][label] = []
                self.data[qid][label].append(feat)
        
        self.data = {k: v for k, v in self.data.items() if len(v) > 1}
        self.qids = list(self.data.keys())
    
    def __allocate_minibatch(self):
        self.features = [np.zeros((MB_SIZE, FEAT_COUNT), dtype=np.float32) for i in range(2)]
        self.labels = np.zeros((MB_SIZE), dtype=np.int64)
        
    def __clear_minibatch(self):
        for i in range(2):
            self.features[i].fill(np.float32(0))
            
    def __next__(self):
        self.__clear_minibatch()
        qids = random.choices(self.qids, k=MB_SIZE)
        for i in range(MB_SIZE):
            labels = random.choices(list(self.data[qids[i]].keys()), k=2)
            labels.sort(reverse=True)
            for j in range(2):
                feats = self.data[qids[i]][labels[j]]
                feat = feats[random.randint(0, len(feats) - 1)]
                for k in range(FEAT_COUNT):
                    self.features[j][i, k] = feat[k]
        
        return [torch.from_numpy(self.features[i]).to(DEVICE) for i in range(2)], torch.from_numpy(self.labels).to(DEVICE)
    
    
class DataReaderTest():
    def __init__(self, data_file):
        self.data_file = data_file

    def __iter__(self):
        self.reader = open(self.data_file, mode='r', encoding="utf-8")
        self.__allocate_minibatch()
        return self
    
    def __allocate_minibatch(self):
        self.features = np.zeros((MB_SIZE, FEAT_COUNT), dtype=np.float32)
        self.labels = np.zeros((MB_SIZE), dtype=np.int64)
        
    def __clear_minibatch(self):
        self.features.fill(np.float32(0))
            
    def __next__(self):
        self.__clear_minibatch()
        qids = []
        labels = []
        cnt = 0
        for i in range(MB_SIZE):
            line = self.reader.readline()
            if line == '':
                raise StopIteration
                break
            
            qid, label, feat = parse_line(line)
            qids.append(qid)
            labels.append(label)
            
            for j in range(FEAT_COUNT):
                self.features[i, j] = feat[j]
            
            cnt += 1
        
        return torch.from_numpy(self.features).to(DEVICE), qids, labels, cnt

### Define the model

In [67]:
class MLP(nn.Module):

    def __init__(
            self,
            input_dim=137,
            index_dim=1,
            hidden_dim=2048,
            activation=nn.ReLU(),
    ):
        super().__init__()
        self.index_dim = index_dim
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.act = activation
        
        self.main = nn.Sequential(
            nn.Linear(input_dim + index_dim, hidden_dim),
            activation,
            nn.LayerNorm(hidden_dim),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(hidden_dim, hidden_dim),
            activation,
            nn.LayerNorm(hidden_dim),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(hidden_dim, hidden_dim),
            activation,
            nn.LayerNorm(hidden_dim),
            nn.Dropout(DROPOUT_RATE),
            nn.Linear(hidden_dim, input_dim),
        )
        

    def forward(self, features, t):
        input = features.view(-1, self.input_dim)
        t = t.view(-1, self.index_dim).float()

        # forward
        h = torch.cat([input, t], dim=1)  # concat
        output = self.main(h)  # forward
        return output.view(-1, self.input_dim)

In [68]:
class Diffusion(nn.Module):
    def __init__(self, model, num_features=137, n_times=1000, beta_minmax=[1e-4, 2e-2], device='cuda'):
        super(Diffusion, self).__init__()

        self.model = model.to(device)
        self.n_features = num_features
        self.n_times = n_times

        # define linear variance schedule(betas)
        beta_1, beta_T = beta_minmax
        betas = torch.linspace(start=beta_1, end=beta_T, steps=n_times).to(device)  # follows DDPM paper
        self.sqrt_betas = torch.sqrt(betas)

        # define alpha for forward diffusion kernel
        self.alphas = 1 - betas
        self.sqrt_alphas = torch.sqrt(self.alphas)
        alpha_bars = torch.cumprod(self.alphas, dim=0)
        self.sqrt_one_minus_alpha_bars = torch.sqrt(1 - alpha_bars)
        self.sqrt_alpha_bars = torch.sqrt(alpha_bars)

        self.device = device

    def extract(self, a, t, x_shape):
        """
            from lucidrains' implementation
                https://github.com/lucidrains/denoising-diffusion-pytorch/blob/beb2f2d8dd9b4f2bd5be4719f37082fe061ee450/denoising_diffusion_pytorch/denoising_diffusion_pytorch.py#L376
        """
        b, *_ = t.shape
        out = a.gather(-1, t)
        return out.reshape(b, *((1,) * (len(x_shape) - 1)))

    def scale_to_minus_one_to_one(self, x):
        # according to the DDPMs paper, normalization seems to be crucial to train reverse process network
        return x * 2 - 1

    def make_noisy(self, x_zeros, t):
        # perturb x_0 into x_t (i.e., take x_0 samples into forward diffusion kernels)
        epsilon = torch.randn_like(x_zeros).to(self.device)

        sqrt_alpha_bar = self.extract(self.sqrt_alpha_bars, t, x_zeros.shape)
        sqrt_one_minus_alpha_bar = self.extract(self.sqrt_one_minus_alpha_bars, t, x_zeros.shape)

        # Let's make noisy sample!: i.e., Forward process with fixed variance schedule
        #      i.e., sqrt(alpha_bar_t) * x_zero + sqrt(1-alpha_bar_t) * epsilon
        noisy_sample = x_zeros * sqrt_alpha_bar + epsilon * sqrt_one_minus_alpha_bar

        return noisy_sample.detach(), epsilon

    def forward(self, x_zeros, y):
        # x_zeros = self.scale_to_minus_one_to_one(x_zeros) # normalize to -1 to 1

        B, _ = x_zeros.shape

        # (1) randomly choose diffusion time-step
        t = torch.randint(low=0, high=self.n_times, size=(B,)).long().to(self.device)

        # (2) forward diffusion process: perturb x_zeros with fixed variance schedule
        perturbed, epsilon = self.make_noisy(torch.cat((x_zeros, y), dim=1), t)

        # (3) predict epsilon(noise) given perturbed data at diffusion-timestep t.
        pred_epsilon = self.model(perturbed, t)

        return perturbed, epsilon, pred_epsilon

    def denoise_at_t(self, x_t, timestep, t):
        B, _ = x_t.shape
        if t > 1:
            z = torch.randn_like(x_t).to(self.device)
        else:
            z = torch.zeros_like(x_t).to(self.device)

        # at inference, we use predicted noise(epsilon) to restore perturbed data sample.
        epsilon_pred = self.model(x_t, timestep)

        alpha = self.extract(self.alphas, timestep, x_t.shape)
        sqrt_alpha = self.extract(self.sqrt_alphas, timestep, x_t.shape)
        sqrt_one_minus_alpha_bar = self.extract(self.sqrt_one_minus_alpha_bars, timestep, x_t.shape)
        sqrt_beta = self.extract(self.sqrt_betas, timestep, x_t.shape)

        # denoise at time t, utilizing predicted noise
        x_t_minus_1 = 1 / sqrt_alpha * (x_t - (1 - alpha) / sqrt_one_minus_alpha_bar * epsilon_pred) + sqrt_beta * z

        return x_t_minus_1.clamp(-1., 1), epsilon_pred
        # return x_t_minus_1, epsilon_pred

    def predict(self, x):
        # start from random noise vector, x_0 (for simplicity, x_T declared as x_t instead of x_T)
        x_t = torch.randn((x.size(0), self.n_features)).to(self.device)

        # autoregressively denoise from x_T to x_0

        # Denoise
        for t in range(self.n_times - 1, -1, -1):
            for j in range(5): ## Harmonization
                timestep = torch.tensor([t]).repeat_interleave(x.size(0), dim=0).long().to(self.device)
                x_t_1_unknown, _ = self.denoise_at_t(x_t, timestep, t)
                if t > 0:
                    x_t_1_known, epsilon = self.make_noisy(x, timestep - 1)
                else:
                    x_t_1_known = x
                    epsilon = torch.zeros_like(x_t_1_known[:, :self.n_features]).to(self.device)
                x_t_1 = torch.cat((x_t_1_known[:, :(self.n_features - 1)], x_t_1_unknown[:, (self.n_features - 1):]), dim=1)
                if j < 4 and t > 0:
                    # Add noise for one step
                    x_t = self.sqrt_alphas[t] * x_t_1 + self.sqrt_betas[t] * epsilon.to(self.device)
                else:
                    x_t = x_t_1

        x_0 = x_t
        return x_0

In [69]:
class WeightedMSELoss(nn.Module):
    def __init__(self, weight):
        super(WeightedMSELoss, self).__init__()
        self.weight = weight

    def forward(self, input, target):
        # Create a weight tensor with the same shape as input/target
        # Set all weights to 1, except for the last feature which is multiplied by the specified weight
        weights = torch.ones_like(input)
        weights[:, -1] = self.weight

        # Compute the squared difference
        squared_diff = (input - target) ** 2

        # Apply the weights and take the mean
        weighted_loss = weights * squared_diff
        return weighted_loss.mean()

### Train and evaluate

In [70]:
READER_TRAIN = DataReaderTest(DATA_FILE_TRAIN)
READER_TRAIN_ITER = iter(READER_TRAIN)
READER_TEST = DataReaderTest(DATA_FILE_TEST)
READER_TEST_ITER = iter(READER_TEST)

model = MLP(input_dim=FEAT_COUNT + 1, hidden_dim=NUM_HIDDEN_NODES)
net = Diffusion(model, num_features=FEAT_COUNT + 1, device=DEVICE)
optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE)
if MODE == 'pairwise':
    criterion = nn.MarginRankingLoss(margin=1.0)
elif MODE == 'pointwise':
    criterion = nn.MSELoss()
    # criterion = WeightedMSELoss(weight=20.0)

In [71]:
iterator_len = 0
for features, _, labels, _ in READER_TRAIN_ITER:
    iterator_len += 1
    
print(f"Data loaded. Train data batch count: {iterator_len}")

Data loaded. Train data batch count: 41


In [72]:
def train(net):
    train_loss = 0.0
    net.train()
    e_size = 0
    
    for _ in range(EPOCH_SIZE):
        for features, _, labels, _ in READER_TRAIN_ITER:
            e_size += 1
            #Read in a new mini-batch of data!
            # features, labels = next(READER_TRAIN_ITER) #Read in a new mini-batch of data!
            labels = torch.tensor(labels).to(DEVICE)
            labels = labels.unsqueeze(1)
            
            if MODE == 'pairwise':
                noisy_input0, epsilon0, pred_epsilon0 = net(features[0], labels)
                noisy_input1, epsilon1, pred_epsilon1 = net(features[1], labels)
                loss = ...
            elif MODE == 'pointwise':
                noisy_input, epsilon, pred_epsilon = net(features, labels)
                # noisy_input, epsilon, pred_epsilon = net(features[0], labels)
                loss = criterion(pred_epsilon, epsilon)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            break
            
    return train_loss / e_size
    


def test(net, epoch, train_loss):
    net.eval()
    reader_test_iter = iter(READER_TEST)
    c = False
    
    with torch.no_grad():
        results = {}
        for features, qids, labels, cnt in reader_test_iter:
            random_labels = torch.randint(0, 5 if 'MSLR' in DATA_DIR else 3, (cnt, 1)).to(DEVICE)
            out = net.predict(torch.cat((features, random_labels), dim=1))[:, -1]
            out = out.unsqueeze(1).cpu()
            if c:
                print(out)
                print(out.shape)
                c = False
            row_cnt = len(qids)
            for i in range(row_cnt):
                if qids[i] not in results:
                    results[qids[i]] = []
                results[qids[i]].append((labels[i], out[i][0]))
            break
        
        avgp = 0
        avgndcg = 0
        for qid, docs in results.items():
            dcg = 0
            ranked = sorted(docs, key=lambda x: x[1], reverse=True)

            relevant_in_top10 = sum(1 for doc in ranked[:10] if doc[0] > 0)
            p = relevant_in_top10 / min(10, len(ranked))
            avgp += p
            
            for i in range(min(10, len(ranked))):
                rank = i + 1
                label = ranked[i][0]
                dcg += ((2**label - 1) / math.log2(rank + 1))
            idcg = 0
            ranked = sorted(docs, key=lambda x: x[0], reverse=True)
            for i in range(min(10, len(ranked))):
                rank = i + 1
                label = ranked[i][0]
                idcg += ((2**label - 1) / math.log2(rank + 1))
            if idcg > 0:
                avgndcg += (dcg / idcg)
        avgp /= len(results)
        avgndcg /= len(results)
        print(f'epoch:{epoch}, loss: {train_loss}, avgp: {avgp}, avgndcg: {avgndcg}')
        
    return avgp, avgndcg

In [73]:
def test_a_c(net, reader_data):
    net.eval()
    total_mse = 0
    total_samples = 0
    
    with torch.no_grad():
        for features, _, labels, cnt in reader_data:
            labels = torch.tensor(labels).to(DEVICE).unsqueeze(1)
            input_data = torch.cat((features, labels), dim=1)
            
            # Sample random timesteps
            t = torch.randint(0, net.n_times, (cnt,), device=DEVICE)
            
            # Add noise
            noisy_input, epsilon = net.make_noisy(input_data, t)
            
            # Predict epsilon
            pred_epsilon = net.model(noisy_input, t)
            
            # Calculate MSE
            mse = F.mse_loss(pred_epsilon, epsilon, reduction='sum')
            total_mse += mse.item()
            total_samples += cnt
            break
    
    avg_mse = total_mse / total_samples / (FEAT_COUNT + 1)
    return avg_mse


def test_a_c_by_individual_features(net, reader_data):
    net.eval()
    total_mse = 0
    total_samples = 0
    avg_mses = {}
    
    with torch.no_grad():
        for features, _, labels, cnt in reader_data:
            labels = torch.tensor(labels).to(DEVICE).unsqueeze(1)
            input_data = torch.cat((features, labels), dim=1)
            
            # Sample random timesteps
            t = torch.randint(0, net.n_times, (cnt,), device=DEVICE)
            
            # Add noise
            noisy_input, epsilon = net.make_noisy(input_data, t)
            
            # Predict epsilon
            pred_epsilon = net.model(noisy_input, t)
            
            # Calculate MSE
            for i in range(FEAT_COUNT+1):
                mse = F.mse_loss(pred_epsilon[:, i], epsilon[:, i], reduction='sum')
                total_mse += mse.item()
                avg_mses[i] = mse.item() / cnt
            
            total_samples += cnt
    
    for i in range(FEAT_COUNT+1):
        avg_mses[i] /= total_samples
        avg_mses[i] /= (FEAT_COUNT + 1)
        
    return avg_mses


def test_b_d(net, reader_data):
    net.eval()
    total_mse = 0
    total_samples = 0
    
    with torch.no_grad():
        for features, _, labels, cnt in reader_data:
            labels = torch.tensor(labels).to(DEVICE).unsqueeze(1)
            input_data = torch.cat((features, labels), dim=1)
            
            predicted_data = net.predict(input_data)
            
            # Calculate MSE
            mse = F.mse_loss(predicted_data, input_data, reduction='sum')
            total_mse += mse.item()
            total_samples += cnt
            break
    
    avg_mse = total_mse / total_samples
    return avg_mse


def test_b_d_over_noises(net, reader_data):
    net.eval()
    total_mse = 0
    total_samples = 0
    noise_mse = 0
    
    with torch.no_grad():
        for features, _, labels, cnt in reader_data:
            labels = torch.tensor(labels).to(DEVICE).unsqueeze(1)
            input_data = torch.cat((features, labels), dim=1)
            
            # start from random noise vector, x_0 (for simplicity, x_T declared as x_t instead of x_T)
            x_t = torch.randn((input_data.size(0), net.n_features)).to(net.device)

            # autoregressively denoise from x_T to x_0

            # Denoise
            for t in range(net.n_times - 1, -1, -1):
                for j in range(5): ## Harmonization
                    timestep = torch.tensor([t]).repeat_interleave(input_data.size(0), dim=0).long().to(net.device)
                    x_t_1_unknown, epsilon_pred = net.denoise_at_t(x_t, timestep, t)
                    if t > 0:
                        x_t_1_known, epsilon = net.make_noisy(input_data, timestep - 1)
                        last_epsilon = epsilon
                        last_epsilon_pred = epsilon_pred
                    else:
                        x_t_1_known = input_data
                        epsilon = torch.zeros_like(x_t_1_known[:, :net.n_features]).to(net.device)
                    x_t_1 = torch.cat((x_t_1_known[:, :(net.n_features - 1)], x_t_1_unknown[:, (net.n_features - 1):]), dim=1)
                    if j < 4 and t > 0:
                        # Add noise for one step
                        x_t = net.sqrt_alphas[t] * x_t_1 + net.sqrt_betas[t] * epsilon.to(net.device)
                    else:
                        x_t = x_t_1

            predicted_data = x_t
            
            # Calculate MSE
            mse = F.mse_loss(predicted_data, input_data, reduction='sum')
            total_mse += mse.item()
            total_samples += cnt
            
            mse = F.mse_loss(last_epsilon_pred, last_epsilon, reduction='mean')
            noise_mse += mse.item()
            
            break
    
    avg_mse = total_mse / total_samples / (FEAT_COUNT + 1)
    avg_noise_mse = noise_mse / total_samples
    return avg_mse, avg_noise_mse


def test_label_prediction(net, reader_data):
    net.eval()
    total_mse = 0
    total_samples = 0
    
    with torch.no_grad():
        for features, _, labels, cnt in reader_data:
            # Convert labels to tensor and reshape
            true_labels = torch.tensor(labels, device=DEVICE).float().unsqueeze(1)
            
            # Generate random noise for labels
            noisy_labels = torch.randn(cnt, 1, device=DEVICE)
            
            # Combine features and noisy labels
            input_data = torch.cat((features, noisy_labels), dim=1)
            
            # Use the model to predict (in-paint) the labels
            predicted_data = net.predict(input_data)
            
            # Extract the predicted labels (last column)
            predicted_labels = predicted_data[:, -1].unsqueeze(1)
            
            # Calculate MSE for labels only
            mse = F.mse_loss(predicted_labels, true_labels, reduction='sum')
            total_mse += mse.item()
            total_samples += cnt
            break
    
    avg_mse = total_mse / total_samples
    return avg_mse

In [74]:
print('Dataset: {}'.format(DATA_DIR))
print(f"Model has {(sum(p.numel() for p in model.parameters() if p.requires_grad)):,} trainable parameters")
print('Learning rate: {}'.format(LEARNING_RATE))
print('Mode: {} - Generative'.format(MODE))

train_loss = 'n/a'
for epoch in range(NUM_EPOCHS):
    avgp, avgndcg = test(net, epoch, str(train_loss))
    a = test_a_c(net, READER_TRAIN_ITER)
    # b = test_b_d(net, READER_TRAIN_ITER)
    c = test_a_c(net, READER_TEST_ITER)
    # d = test_b_d(net, READER_TEST_ITER)
    e = test_label_prediction(net, READER_TRAIN_ITER)
    f = test_label_prediction(net, READER_TEST_ITER)
    
    avg_mses_over_features = test_a_c_by_individual_features(net, READER_TRAIN_ITER)
    b, b_noise = test_b_d_over_noises(net, READER_TRAIN_ITER)
    d, d_noise = test_b_d_over_noises(net, READER_TEST_ITER)
    
    log_data = {
        'train_loss': train_loss,
        'avgp': avgp,
        'avgndcg': avgndcg,
        'a': a,
        'b': b,
        'b_noise': b_noise,
        'c': c,
        'd': d,
        'd_noise': d_noise,
        'e': e,
        'f': f,
    }
    for i in range(FEAT_COUNT+1):
        log_data[f'feat_{i}_train'] = avg_mses_over_features[i]
    
    wandb.log(log_data)

    train_loss = train(net)
    
wandb.finish()
# save model
torch.save(net.state_dict(), MODEL_SAVE_PATH)

print('Finished training')

Dataset: ../data/MQ2007/Fold1/
Model has 14,895 trainable parameters
Learning rate: 0.0002
Mode: pointwise - Generative
epoch:0, loss: n/a, avgp: 0.223076923076923, avgndcg: 0.2372077225205562
epoch:1, loss: 1.0119218070758507, avgp: 0.23846153846153847, avgndcg: 0.21467299000989723
epoch:2, loss: 0.9999667703814339, avgp: 0.22307692307692306, avgndcg: 0.2562327052999226
epoch:3, loss: 0.9999715257727075, avgp: 0.23461538461538461, avgndcg: 0.23792936333775658
epoch:4, loss: 0.9999251628760248, avgp: 0.19615384615384618, avgndcg: 0.1777696552933987
epoch:5, loss: 0.9913653177791275, avgp: 0.25384615384615383, avgndcg: 0.30464640509380525
epoch:6, loss: 0.9823527331755031, avgp: 0.22692307692307684, avgndcg: 0.31761629675471337
epoch:7, loss: 0.9684183885692619, avgp: 0.2846153846153846, avgndcg: 0.30743045907692956
epoch:8, loss: 0.9652162569691427, avgp: 0.27692307692307694, avgndcg: 0.3239076408929834
epoch:9, loss: 0.9510836318368092, avgp: 0.27307692307692305, avgndcg: 0.2859069372

a,█▃▃▃▃▃▂▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
avgndcg,▂▂▃▂▁▄▄▃▆▅▆▆▆▅▅▆▇█▅▆██▇▆█▆▆▆██▇▅▆▇▇▆▆▇▆▆
avgp,▂▃▂▃▁▅▄▄▅▆▇▇▅▅▅▅▆▇▄▅█▆▆▆▇▆▄▆▅█▆▅▆▆▆▆▇▇▅▆
b,▇▇██▆▃▄▄▃▂▂▂▁▁▂▁▂▁▂▁▁▁▁▂▁▂▁▂▁▁▁▁▁▁▁▂▁▁▁▁
b_noise,█▁▁▁▁▂▁▁▁▁▁▁▁▂▁▁▁▁▁▁▁▂▂▁▂▂▂▂▁▂▂▂▂▂▂▁▂▂▂▂
c,█▃▃▃▃▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
d,█▇█▇▇▅▄▄▄▃▂▂▂▂▁▂▂▁▂▁▂▁▁▁▂▁▂▂▂▁▂▁▂▂▂▂▁▁▁▁
d_noise,█▁▁▁▁▁▁▁▁▁▁▁▁▁▂▁▂▂▂▁▂▁▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▁
e,██▇█▅▄▃▃▃▂▂▁▁▁▂▂▂▁▂▂▁▁▁▂▁▂▁▂▁▁▂▁▂▂▁▂▁▁▁▁
f,█▇██▇▅▄▄▄▃▂▂▂▁▂▂▂▁▂▂▁▂▁▂▁▂▁▁▁▁▂▂▂▂▂▂▁▁▁▂
feat_0_train,█▁▃▂▁▂▂▁▂▃▂▂▂▂▂▃▁▂▃▂▁▁▁▂▁▃▁▂▃▂▃▂▃▂▂▂▂▂▄▂


Finished training


In [75]:
# wandb.finish()